# 第 3 章 · `MemoryManager`（可执行源码副本）

对照：`../hermes_src/agent/memory_manager.py`

**讲解要点**：`builtin` 可多个逻辑位；**外部 Provider 同时只允许一个**。

```mermaid
flowchart TD
    MM[MemoryManager] -->|accept| B[builtin]
    MM -->|accept first| E1[external A]
    MM -.->|reject| E2[external B]
```

> 先 Run 第 2 章定义 `MemoryProvider` / `DemoProvider`，或在本 notebook 重新定义精简版。

In [ ]:
# 若未跑第 2 章，先准备最小 ABC + Demo（与第 2 章同构）
from __future__ import annotations
from abc import ABC, abstractmethod
from typing import Any, Dict, List, Optional
import json
import logging

logger = logging.getLogger("memory_manager_demo")
logging.basicConfig(level=logging.INFO)


class MemoryProvider(ABC):
    @property
    @abstractmethod
    def name(self) -> str: ...
    @abstractmethod
    def is_available(self) -> bool: ...
    @abstractmethod
    def initialize(self, session_id: str, **kwargs) -> None: ...
    def system_prompt_block(self) -> str: return ""
    def prefetch(self, query: str, *, session_id: str = "") -> str: return ""
    def queue_prefetch(self, query: str, *, session_id: str = "") -> None: ...
    def sync_turn(self, user_content: str, assistant_content: str, **kwargs) -> None: ...
    @abstractmethod
    def get_tool_schemas(self) -> List[Dict[str, Any]]: ...
    def handle_tool_call(self, tool_name: str, args: Dict[str, Any], **kwargs) -> str:
        raise NotImplementedError


class DemoProvider(MemoryProvider):
    def __init__(self, name: str = "demo"):
        self._name = name
        self._facts: List[str] = []
        self._queued = ""
    @property
    def name(self) -> str: return self._name
    def is_available(self) -> bool: return True
    def initialize(self, session_id: str, **kwargs) -> None: pass
    def system_prompt_block(self) -> str:
        return f"[{self.name}] external memory active"
    def prefetch(self, query: str, *, session_id: str = "") -> str:
        q = (self._queued or query).lower()
        hits = [f for f in self._facts if any(t in f.lower() for t in q.split() if len(t) > 1)] or self._facts[-3:]
        return ("## Recalled\n" + "\n".join(f"- {h}" for h in hits)) if hits else ""
    def queue_prefetch(self, query: str, *, session_id: str = "") -> None:
        self._queued = query
    def sync_turn(self, user_content: str, assistant_content: str, **kwargs) -> None:
        if user_content and user_content not in self._facts and len(user_content) < 120:
            self._facts.append(user_content)
    def get_tool_schemas(self) -> List[Dict[str, Any]]:
        return [{"name": f"{self.name}_store", "description": "store", "parameters": {"type": "object", "properties": {"fact": {"type": "string"}}, "required": ["fact"]}}]
    def handle_tool_call(self, tool_name: str, args: Dict[str, Any], **kwargs) -> str:
        self._facts.append(str(args.get("fact", "")))
        return json.dumps({"ok": True, "facts": self._facts}, ensure_ascii=False)


class BuiltinProvider(MemoryProvider):
    """占位：真 Hermes 的 builtin 名称必须是 'builtin' 才会绕过单外部限制。"""
    @property
    def name(self) -> str: return "builtin"
    def is_available(self) -> bool: return True
    def initialize(self, session_id: str, **kwargs) -> None: pass
    def system_prompt_block(self) -> str: return "[builtin] MEMORY.md / USER.md layer"
    def get_tool_schemas(self) -> List[Dict[str, Any]]: return []

print("providers ready")

## ① MemoryManager 源码副本（核心方法）

In [ ]:
# =============================================================================
# SOURCE COPY: hermes_src/agent/memory_manager.py :: MemoryManager
# 保留：单外部限制、build_system_prompt、prefetch_all、sync_all
# 省略：线程池后台 sync、skill scaffolding 剥离、core tool 影子检测
# =============================================================================

class MemoryManager:
    """Orchestrates builtin plus at most one external provider."""

    def __init__(self) -> None:
        self._providers: List[MemoryProvider] = []
        self._tool_to_provider: Dict[str, MemoryProvider] = {}
        self._has_external: bool = False

    def add_provider(self, provider: MemoryProvider) -> None:
        """Register a memory provider.

        Built-in provider (name \"builtin\") is always accepted.
        Only **one** external (non-builtin) provider is allowed — a second
        attempt is rejected with a warning.
        """
        is_builtin = provider.name == "builtin"

        if not is_builtin:
            if self._has_external:
                existing = next(
                    (p.name for p in self._providers if p.name != "builtin"), "unknown"
                )
                logger.warning(
                    "Rejected memory provider '%s' — external provider '%s' is "
                    "already registered. Only one external memory provider is "
                    "allowed at a time. Configure which one via memory.provider "
                    "in config.yaml.",
                    provider.name, existing,
                )
                return
            self._has_external = True

        self._providers.append(provider)

        for schema in provider.get_tool_schemas():
            tool_name = schema.get("name") if isinstance(schema, dict) else None
            if tool_name and tool_name not in self._tool_to_provider:
                self._tool_to_provider[tool_name] = provider

        logger.info(
            "Memory provider '%s' registered (%d tools)",
            provider.name,
            len(provider.get_tool_schemas()),
        )

    def build_system_prompt(self) -> str:
        blocks = []
        for provider in self._providers:
            try:
                block = provider.system_prompt_block()
                if block and block.strip():
                    blocks.append(block)
            except Exception as e:
                logger.warning("provider %s system_prompt_block failed: %s", provider.name, e)
        return "\n\n".join(blocks)

    def prefetch_all(self, query: str, *, session_id: str = "") -> str:
        parts = []
        for provider in self._providers:
            try:
                result = provider.prefetch(query, session_id=session_id)
                if result and result.strip():
                    parts.append(result)
            except Exception as e:
                logger.debug("prefetch failed %s: %s", provider.name, e)
        return "\n\n".join(parts)

    def sync_all(self, user_content: str, assistant_content: str, **kwargs) -> None:
        for provider in self._providers:
            try:
                provider.sync_turn(user_content, assistant_content, **kwargs)
            except Exception as e:
                logger.debug("sync failed %s: %s", provider.name, e)

    def queue_prefetch_all(self, query: str, *, session_id: str = "") -> None:
        for provider in self._providers:
            try:
                provider.queue_prefetch(query, session_id=session_id)
            except Exception as e:
                logger.debug("queue_prefetch failed %s: %s", provider.name, e)

    def handle_tool_call(self, tool_name: str, args: dict) -> str:
        p = self._tool_to_provider.get(tool_name)
        if not p:
            return json.dumps({"error": f"no provider for {tool_name}"})
        return p.handle_tool_call(tool_name, args)

    @property
    def providers(self):
        return list(self._providers)


print("MemoryManager 已定义")

## ② 演示：第二个外部 Provider 被拒绝

In [ ]:
mm = MemoryManager()
mm.add_provider(BuiltinProvider())
mm.add_provider(DemoProvider("honcho"))
mm.add_provider(DemoProvider("mem0"))  # 应被 reject

print("registered =", [p.name for p in mm.providers])
print("has_external =", mm._has_external)
print("\nSP:\n", mm.build_system_prompt())

mm.sync_all("我在准备 Agent Runtime 面试", "收到")
mm.queue_prefetch_all("面试")
print("\nprefetch:\n", mm.prefetch_all("占位"))